# Exercise 9: series

* pandas series vs numpy arrays [explanation](https://jakevdp.github.io/PythonDataScienceHandbook/03.01-introducing-pandas-objects.html)

### Common series operations
These are the most common series operations we use. Refer to the `pandas` docs for even more!

* Getting dates, hours, minutes from datetime types (`df.datetime_col.dt.date`)
* Parsing strings (`df.string_col.str.split('_')`)

### Common geoseries operations
These are the most common. Refer to the `geopandas` docs for even more!

* `distance` between 2 points or a point to a polygon or line [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.distance.html)
* `intersects`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.intersects.html)
* `within`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.within.html)
* `contains`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.contains.html)

In fact, we've often used geoseries methods without even realizing it. Often, we'd create a new column that stores either the line's length or a polygon's area. `gdf.geometry` is a geoseries, and we call methods on that geoseries, and add that as a new column.

For calculations like `length`, `area`, and `distance`, we need to use a projected CRS that has units like meters or feet. We cannot use decimal degrees (do not use WGS 84 / EPSG:3326)! Distance calculations must be done only once the spherical 3D Earth has been converted into a 2D plane.

* `length`: get the length of a line (`gdf.geometry.length`)
* `area`: get the area of a polygon (`gdf.geometry.area`)
* `centroid`: get the centroid of a polygon (`gdf.geometry.centroid`)
* `x`: get the x coordinate of a point (`gdf.geometry.x`)
* `y`: get the y coordinate of a point (`gdf.geometry.y`)

### Arrays
* Occasionally, we may even use arrays, especially when the datasets get even larger but we have simple mathematical calculations
* If we need to apply an exponential decay function to a distance column, we essentially want to multiple `distance` by some number
* Since this exponential decay function is somewhat custom and requires us to write our own formula, we would extract the column as a series (`df.distance`) and multiply each value by some other number.
* Even quicker is to use `numpy` with `distance_array = np.array(df.distance)` and get `exponential_array = distance_array*some_number`

In [4]:
import geopandas as gpd
import intake
import numpy as np
import pandas as pd

catalog = intake.open_catalog(
    "../_shared_utils/shared_utils/shared_data_catalog.yml")

In [5]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

If you're asking how far is a transit stop from the interstate, you'd want the distance of every point (every row) compared to an interstate highway geometry.

Let's prep the datasets to use series / geoseries to do this.

In [6]:
#stops = catalog.ca_transit_stops.read()[["agency", "stop_id", 
                                         #"stop_name", "geometry"]]
highways = catalog.state_highway_network.read()

In [7]:
#stops.shape, type(stops)

In [8]:
#stops.sample()

In [9]:
#highways.shape, type(highways)

Since we want to know the distance from a stop's point to the interstate generally, we need a dissolve. We don't want to compare the distance against the I-5, the I-10 individually, but to the interstate system as a whole.

In [10]:
interstates = (highways[highways.RouteType=="Interstate"]
               .dissolve()
               .reset_index()
               [["geometry"]]
              )

In [11]:
interstates.shape

(1, 1)

In [12]:
# This is still a gdf, just with 1 column
type(interstates)

geopandas.geodataframe.GeoDataFrame

In [13]:
# Pulling out the individual column, it becomes a series/geoseries.
# It's a geoseries here because we had a gdf. 
# If it was a df, it would be a series.
#print(type(stops.geometry))
#print(type(interstates.geometry))

Distance is something you can calculate using `geopandas`.

Specifically, it takes a geoseries on the left, and either a geoseries or a single geometry on the right.

An example of having 2 geoseries would be comparing the distance between 2 points. On the left, it would be a geoseries of the origin points and on the right, destination points.

In [14]:
# We get a warning if we leave it in EPSG:4326!
# stops.geometry.distance(interstates.geometry.iloc[0])

In [15]:
# stops_geom = stops.to_crs("EPSG:2229").geometry

In [16]:
# type(stops_geom)

In [17]:
# stops_geom.head()

In [18]:
# interstates_geom = interstates.to_crs("EPSG:2229").geometry.iloc[0]
# distance_series = stops_geom.distance(interstates_geom)

In [19]:
# distance_series = stops_geom.distance(interstates_geom)

In [20]:
# distance_series.head()

In [21]:
# Let's make sure that for every stop, a distance is calculated
# print(f"# rows in stops: {len(stops_geom)}")
# print(f"# rows in stops: {len(distance_series)}")

In [22]:
# distance is numeric, not a geometry, so we're back to being a series
# type(distance_series)

What can we do with this? 

We usually add it as a new column. Since we did nothing to shift the index, we can just attach the series back to our gdf.

Getting a distance calculation using geoseries is much quicker than a row-wise lambda function where you calculate the distance.

```
Alternative method that's slower:
      
interstate_geom = interstates.geometry.iloc[0]

stops = stops.assign(
   distance = stops.geometry.apply(
         lambda x: x.distance(interstate_geom))
)   
```

In [23]:
#stops = stops.assign(
#    distance_to_interstate = distance_series
#)

In [24]:
#%%timeit
#distance_series = stops_geom.distance(interstates_geom)

In [25]:
"""
%%timeit
stops.assign(
   distance = stops.geometry.apply(
         lambda x: x.distance(interstates_geom))
)  """

'\n%%timeit\nstops.assign(\n   distance = stops.geometry.apply(\n         lambda x: x.distance(interstates_geom))\n)  '

In [26]:
"""
import dask_geopandas as dg

stops_gddf = dg.from_geopandas(stops, npartitions=2)
stops_geom_dg = stops_gddf.to_crs("EPSG:2229").geometry """

'\nimport dask_geopandas as dg\n\nstops_gddf = dg.from_geopandas(stops, npartitions=2)\nstops_geom_dg = stops_gddf.to_crs("EPSG:2229").geometry '

In [27]:
 """ %%timeit

distance_series = stops_geom_dg.distance(interstates_geom)"""

' %%timeit\n\ndistance_series = stops_geom_dg.distance(interstates_geom)'

## To Do
* Use the `stop_times` table and `stops` table.
* Calculate the straight line distance between the first and last stop for each trip. Call this column `trip_distance`
* Calculate the distance between each stop to the nearest interstate. For each trip, keep the value for the stop that's the closest to the interstate. Call this column `shortest_distance_hwy`.
* For each trip, add these 2 new columns, but use series, geoseries, and/or arrays to assign it.
* Provide a preview of the resulting df (do not export)

In [28]:
GCS_FILE_PATH = ("gs://calitp-analytics-data/data-analyses/"
                 "rt_delay/compiled_cached_views/"
                )

analysis_date = "2023-01-18"
STOP_TIMES_FILE = f"{GCS_FILE_PATH}st_{analysis_date}.parquet"
STOPS_FILE = f"{GCS_FILE_PATH}stops_{analysis_date}.parquet"

highways = catalog.state_highway_network.read()

### Prep
#### Highways -> Interstate Only

In [29]:
def find_interstate(highways_df):
    """
    Get one row for each route-county combo
    """
    df = (highways_df[highways_df.RouteType=="Interstate"]).reset_index(drop = True)
    df = highways_df[['Route','County','geometry']].dissolve(by = ['Route','County']).reset_index()
    return df

In [30]:
interstates.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

#### Prep Stops

In [31]:
def prep_stops():
    
    df = gpd.read_parquet(STOPS_FILE)
    df = df[['feed_key', 'stop_id', 'geometry']]
    
    # Change to feet
    df = df.to_crs("EPSG:2229")
    
    return df

In [32]:
all_stops = prep_stops()

In [33]:
all_stops.stop_id.nunique()

51514

In [34]:
all_stops.shape

(84688, 3)

### Calculate the straight line distance between the first and last stop for each trip. Call this column trip_distance

In [35]:
def prep_stoptimes():
    df = pd.read_parquet(STOP_TIMES_FILE)
    
    # Only keep sequence
    df = df[['feed_key','trip_id','stop_id','stop_sequence']]
    
    # Sort so sequence goes from lowest to highest value
    df = df.sort_values(by = ['feed_key','trip_id','stop_sequence']).reset_index(drop = True)
    return df 

In [36]:
stoptimes = prep_stoptimes()

In [37]:
stoptimes.stop_id.nunique()

49560

In [38]:
stoptimes.shape

(3589931, 4)

In [39]:
#_merge results
#both          3589718
#left_only        3670
#right_only        263

# pd.merge(stops, stop_times, how = 'outer', on = ['feed_key', 'stop_id'], indicator = True)[['_merge']].value_counts()

In [40]:
def complete_stop_info():
    stop_times = prep_stoptimes()
    stops = prep_stops()
    
    m1 = pd.merge(stops, stop_times, how = 'inner', on = ['feed_key', 'stop_id'])
    
    return m1

In [41]:
stop_w_times = complete_stop_info()

In [42]:
# Trip IDS are slightly different
# stops_m1.head(10)

In [43]:
def first_last_trip():
    m1 = complete_stop_info()
    
    sort_values = ['feed_key','trip_id','stop_sequence']
    duplicate_values = ['feed_key','trip_id',]
    
    # Find the FIRST stop of every trip
    first = (m1
                  .sort_values(by = sort_values)
                  .drop_duplicates(subset = duplicate_values)
                  .reset_index(drop = True)
                 )
    # Find the LAST stop of every trip
    last = (m1
                 .sort_values(by = sort_values,
                              ascending = [True, True, False])
                 .drop_duplicates(subset = duplicate_values)
                 .reset_index(drop = True)
                )
    return first,last

In [45]:
first_stop, last_stop = first_last_trip()

In [46]:
len(first_stop), first_stop.stop_id.nunique()

(109116, 3405)

In [47]:
# strange that the first sequence is not 1...
first_stop.stop_sequence.value_counts().head()

1     94213
0     13073
2      1297
10      114
25      105
Name: stop_sequence, dtype: int64

In [48]:
# first_stop.loc[first_stop.stop_sequence == 21]

In [49]:
# Check out the stop sequences that aren't 1 for one trip...
# stop_times.loc[stop_times.trip_id == "DX12S1213113"]

* Attach distance last_first back to dataframe

In [50]:
def find_distance():
    first_stop, last_stop = first_last_trip()
    
    # Turn the last and first stop into a geo series
    first_stop_series = first_stop.geometry
    last_stop_series = last_stop.geometry
    
    # Find distance of last stop to first
    trip_distance = last_stop_series.distance(first_stop_series)
    
    # Attach the trip_distance series to firststop
    first_stop = first_stop.assign(
    distance_last_first_stop = trip_distance/5_280)
    
    first_stop = first_stop[['feed_key', 'stop_id','trip_id','distance_last_first_stop']]
    return first_stop

In [51]:
trips_w_dist = find_distance()

In [52]:
trips_w_dist.shape

(109116, 4)

In [53]:
all_stops.stop_id.nunique(), trips_w_dist.stop_id.nunique()

(51514, 3405)

In [69]:
stop_w_times.shape


(3589718, 5)

In [55]:
stop_w_times_tripid = set(stop_w_times.trip_id.unique().tolist())
trips_w_dist_trip_id = set(trips_w_dist.trip_id.unique().tolist())
stop_w_times_tripid - trips_w_dist_trip_id

set()

In [58]:
# Attach distance last to first stop by trip_id and feed_key
pd.merge(trips_w_dist, stop_w_times, on = ['feed_key','trip_id'], how = 'outer', indicator = True)[['_merge']].value_counts()

_merge    
both          3589718
left_only           0
right_only          0
dtype: int64

In [71]:
m1 = pd.merge(trips_w_dist, stop_w_times, on = ['feed_key','trip_id'], how = 'inner')

In [73]:
m1 = m1.drop(columns = ['stop_id_x','geometry','stop_sequence'])

In [75]:
m1 = m1.rename(columns = {'stop_id_y':'stop_id'})

In [76]:
m1.sample(3)

,feed_key,trip_id,distance_last_first_stop,stop_id
1067915,57d7a160e4588225238b330da8453912,11077887_M11,6.37,17383
94037,04a2bb9727f58083d3622ca13e5ab97c,t_74580_b_78393_tn_6,1.33,20357
3405117,f0b28ce2e70bba09a56e0b6a25268666,212,0.00,598


In [77]:
m1.shape

(3589718, 4)

### Calculate the distance between each stop to the nearest interstate. For each trip, keep the value for the stop that's the closest to the interstate

In [78]:
interstates_geom = interstates.to_crs("EPSG:2229").geometry.iloc[0]

In [79]:
all_stops_series = all_stops.geometry

In [80]:
distance_series = all_stops_series.distance(interstates_geom)

In [114]:
# Attach distance
all_stops = all_stops.assign(
    distance_stop_interstate = distance_series/5_280)

In [115]:
all_stops.shape

(84688, 4)

### Combine...

In [116]:
m2 = m1.drop_duplicates().reset_index(drop = True)

In [117]:
m2.shape

(3568018, 4)

In [118]:
# Contains distances for every feedkey-trip-stop combo
m2.head()

,feed_key,trip_id,distance_last_first_stop,stop_id
0,008d5112a7e531d0562d26e34d77869d,1080037,8.65,985
1,008d5112a7e531d0562d26e34d77869d,1080037,8.65,987
2,008d5112a7e531d0562d26e34d77869d,1080037,8.65,3199
3,008d5112a7e531d0562d26e34d77869d,1080037,8.65,5329
4,008d5112a7e531d0562d26e34d77869d,1080037,8.65,982


In [119]:
 m2.stop_id.nunique()

49551

In [120]:
all_stops.head()

,feed_key,stop_id,geometry,distance_stop_interstate
0,6adf6cd9b6d24ab4ee8ee220e3697a73,15193,POINT (6406071.919 1893385.031),2.30
1,6adf6cd9b6d24ab4ee8ee220e3697a73,14025,POINT (6473074.964 1799088.867),0.46
2,6adf6cd9b6d24ab4ee8ee220e3697a73,15638,POINT (6469235.150 1847678.064),2.20
3,6adf6cd9b6d24ab4ee8ee220e3697a73,10244,POINT (6485154.017 1823944.821),1.43
4,6adf6cd9b6d24ab4ee8ee220e3697a73,20206,POINT (6481746.099 1875962.898),0.35


In [121]:
# Contains distance from interstate for each stop
all_stops.head()

,feed_key,stop_id,geometry,distance_stop_interstate
0,6adf6cd9b6d24ab4ee8ee220e3697a73,15193,POINT (6406071.919 1893385.031),2.30
1,6adf6cd9b6d24ab4ee8ee220e3697a73,14025,POINT (6473074.964 1799088.867),0.46
2,6adf6cd9b6d24ab4ee8ee220e3697a73,15638,POINT (6469235.150 1847678.064),2.20
3,6adf6cd9b6d24ab4ee8ee220e3697a73,10244,POINT (6485154.017 1823944.821),1.43
4,6adf6cd9b6d24ab4ee8ee220e3697a73,20206,POINT (6481746.099 1875962.898),0.35


In [122]:
pd.merge(all_stops, m2, on = ['feed_key', 'stop_id'], how = 'outer',indicator = True)[['_merge']].value_counts()

_merge    
both          3568059
left_only        3670
right_only          0
dtype: int64

In [129]:
m3 = pd.merge(all_stops,m2, on = ['feed_key', 'stop_id'], how = 'inner')

In [130]:
len(m3.drop_duplicates())

3568059

In [131]:
len(m3)

3568059

In [132]:
# Keep only the nearest stop to interstate for each trip
m3 = (m3
      .sort_values(['feed_key','trip_id','distance_stop_interstate'])
      .drop_duplicates(subset = ['feed_key','trip_id'])
      .reset_index(drop = True)
     )

In [133]:
# Test w one trip
# m3_test = m3.loc[m3.trip_id == "10165002002053-DEC22"].reset_index(drop = True)

In [137]:
# Check against original merge before all this manipulation
m3.trip_id.nunique(), stop_w_times.trip_id.nunique()

(100876, 100876)

In [139]:
m3_tripid = set(m3.trip_id.unique().tolist())
stop_w_times_tripid - m3_tripid

set()

In [138]:
m3.sample()

,feed_key,stop_id,geometry,distance_stop_interstate,trip_id,distance_last_first_stop
17379,321ca30d23aae541a30db6263ba24654,760841,POINT (6667607.459 2186303.054),31.99,t_559804_b_33402_tn_0,30.09
